# Mini Exploration 1

We will probe the 8th layer of `gpt2-small`

In [ ]:
!python -m spacy download en_core_web_sm

In [35]:
import torch as t
import spacy
import sklearn
import transformer_lens

from transformer_lens import ( 
    HookedTransformer, HookedTransformerConfig, ActivationCache, utilities
)

t.manual_seed(42)

device = 'mps' if t.backends.mps.is_available() else 'gpu'

In [ ]:
# Experiment Config

LAYER_idx  = 8

In [25]:
model = HookedTransformer.from_pretrained('gpt2-small')
nlp = spacy.load("en_core_web_sm")

sample_text = "Finding himself in a world reminiscent of late Victorian era England, Zhou Mingrui (now Klein Moretti) searches for a way to go home. After reproducing a ritual that presumably caused his transmigration, he masquerades as a god-like entity and forms an organization known as the 'Tarot Club'. He is also later questioned by the police about his friends’ suicides; it is revealed that the police have mystical powers. People with mystical powers are called Beyonders. Through a series of events, he becomes an official Beyonder (one who is affiliated with the story’s Seven Orthodox Churches), and tries to find out why the original Klein committed suicide while trying to eventually return home. Klein visits the Wright and Hound Pub where he was taken to Blackthorn Security, the Nighthawks headquarters, where he formally accepts his position as a Beyonder, starting with Sequence Nine. While deciding his pathway, he visited a room run by Old Neil, an alchemist and his mentor; upon discovering his understanding of ancient languages and deciphered a book of Emperor Roselle Gustav's transcripts, Klein decided to take the seer pathway granting himself both the abilities of Spirit Vision and Dowsing Rod. Due to the potion effects, Neil warned Klein about overusage and the potion must internalize fully to function. After a few days of clerical work, Klein was assigned by Leonard on his first field mission: to visit a mansion at Foreston Street to rescue Elliot, who was taken hostage. Leonard's abilities managed to subdue the two captors, who both merged into a single monster being known as a Rampager. Klein recalls that he have visited that mansion before, aware that the house may contain clues relating to his history and the notebook. After his first mission, Klein's brother, Benson, just moved in to his mansion to reunite with him and Melissa."

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 8615.16it/s]


Loaded pretrained model gpt2-small into HookedTransformer


### Data Handling

In [ ]:
doc = nlp(sample_text)
lookup = {}

for token in doc:
    lookup[(token.idx, token.idx + len(token))] = token, token.pos_

In [70]:
lookup[(0,7)][1]

'VERB'

In [ ]:
tokens_offsets = model.tokenizer(sample_text, return_offsets_mapping=True)
tokens, offsets = tokens_offsets['input_ids'], tokens_offsets['offset_mapping']

kept_tokens = []

for i, (start, end) in enumerate(offsets):
    if sample_text[start] == " ":
        start += 1

    if (start, end) in lookup:
        kept_tokens.append((i, lookup[(start, end)][1]))

In [ ]:
for pair in kept_tokens[:10]:
    print(model.tokenizer.decode(tokens[pair[0]]),  pair[1])

Finding VERB
 himself PRON
 in ADP
 a DET
 world NOUN
 reminiscent ADJ
 of ADP
 late ADJ
 Victorian ADJ
 era NOUN


In [100]:
dataset = []

for i in range(len(kept_tokens) - 1):
    if kept_tokens[i+1][0] - kept_tokens[i][0] == 1:
        if kept_tokens[i+1][1] == 'NOUN':
            dataset.append((kept_tokens[i][0], 1))
        else:
            dataset.append((kept_tokens[i][0], 0))    
    else:  
        continue

In [ ]:
dataset

['Finding',
 'himself',
 'in',
 'a',
 'world',
 'reminiscent',
 'of',
 'late',
 'Victorian',
 'era',
 'England,']

run model with cache activations

In [ ]:
logits, cache = model.run_with_cache(tokens, remove_batch_dim=False)

Inspect model configuration

In [ ]:
l8 = cache.

torch.Size([1, 383, 50257])

12